# Pickwise — Conversational Shopping Assistant (Project P14)
### Product Search with MCP, RAG and Human Approval

**Team:** _names and roll numbers_

| Member | Contribution (sections, code, experiments) |
|---|---|
| | |
| | |
| | |

## 1. Problem Statement

- **Problem (our own words):** Shoppers describe a *situation* (“something to keep my kids busy at bath time that can get messy”), not keywords, and keyword search fails on such requests. We build an assistant that turns a long request into catalogue searches, recommends the top 3 products with reasons taken **only from the product text**, remembers each shopper's preferences across conversations, and **never adds to cart without the shopper's approval**.
- **Research question:** Does a multi-agent system with structured query rewriting recommend the gold product more often than plain hybrid retrieval on the raw request?
- **What success looks like:** Hit@3 / Recall@10 above the hybrid-on-raw-request baseline (target: _set after the retrieval baseline_); every recommendation reason supported by the product text; **100%** of `add_to_cart` actions approved by the shopper.

Code that is reused across the notebook, the MCP server and the app lives in the small `Pickwise/` package next to this notebook; each section below cites the course lab it builds on.

## 2. Setup

Builds on: Module 4 · Agentic_app (`config.py`, `llm.py`).

On Colab: clone the project repo first so `Pickwise/` and `requirements.txt` are next to the notebook, and install + start Ollama (`curl -fsSL https://ollama.com/install.sh | sh`, `ollama serve &`, `ollama pull qwen3:4b`).

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import random, time
import numpy as np, pandas as pd, torch
from datasets import load_dataset

from Pickwise import config
from Pickwise.data import build_catalogue, split_queries, check_no_leakage
from Pickwise.llm import get_llm, reasoning_trace

SEED = config.SEED
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CONFIG = {k: getattr(config, k) for k in [
    "MODEL", "REASONING", "EMBED_MODEL", "RERANK_MODEL", "DATASET_ID", "N_QUERIES", "DISTRACTOR_RATE",
    "N_DEV", "QUICK_RUN", "QUICK_N", "SEED", "RECURSION_LIMIT", "RISKY_TOOLS"]}
CONFIG["DEVICE"] = DEVICE
pd.Series(CONFIG, name="value").to_frame()

/home/additional/Projects/MCP-RAG-SHOPPPER/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,value
MODEL,qwen3:4b
REASONING,True
EMBED_MODEL,BAAI/bge-small-en-v1.5
RERANK_MODEL,cross-encoder/ms-marco-MiniLM-L-6-v2
DATASET_ID,McAuley-Lab/Amazon-C4
N_QUERIES,1000
DISTRACTOR_RATE,0.03
N_DEV,200
QUICK_RUN,False
QUICK_N,100


**Model check.** `qwen3:4b` must be pulled in Ollama. We run it with `reasoning=True`: the reply in `content` is clean and the thinking goes to `additional_kwargs["reasoning_content"]`. With `reasoning=False`, Qwen3 still thinks but its narration leaks into the answer (measured in Phase 0; see `CURRENT_PROJECT_STATUS.md`).

In [3]:
import ollama
pulled = {m.model for m in ollama.list().models}
assert config.MODEL in pulled, f"run: ollama pull {config.MODEL}"

t = time.time()
reply = get_llm().invoke("In one sentence: what is a bath toy?")
print(f"{config.MODEL} answered in {time.time() - t:.1f}s")
print("answer   :", reply.content)
print("reasoning:", len(reasoning_trace(reply)), "chars (kept out of the answer)")

qwen3:4b answered in 3.0s
answer   : A bath toy is a safe, fun object designed specifically for babies and toddlers to play with during bath time.
reasoning: 1153 chars (kept out of the answer)


## 3. Data

Builds on: the brief §02 (dataset and sampling code).

| | |
|---|---|
| **Queries** | `McAuley-Lab/Amazon-C4`, `test` split (21,223 long requests, each with its gold `item_id`) → **fixed sample of 1,000**, `shuffle(seed=42)` |
| **Split** | first **200 = dev** (tuning only) · remaining **800 = test** (reported once) · `QUICK_RUN`: first 100 of test |
| **Catalogue** | `sampled_item_metadata_1M.jsonl` (1.06M items) → the 1,000 gold items **+ ~3% random distractors** (seed 42) |
| **Index text** | the item's `metadata` string (title + description). **`ori_review` is never used**: it is the review the query was written from (answer leakage) |

The catalogue is streamed with `load_dataset` on the first run and cached to `data/catalogue.parquet` (keyed on seed, rate and gold set), so re-runs skip the 640 MB scan.

In [4]:
queries = load_dataset(config.DATASET_ID)["test"].shuffle(seed=SEED).select(range(config.N_QUERIES))
print(queries)
words = pd.Series([len(q.split()) for q in queries["query"]])
print(f"query length: mean {words.mean():.0f} words, median {words.median():.0f}, max {words.max()}")
pd.DataFrame(queries.select(range(3)))[["qid", "query", "item_id"]]

Dataset({
    features: ['qid', 'query', 'item_id', 'user_id', 'ori_rating', 'ori_review'],
    num_rows: 1000
})
query length: mean 43 words, median 36, max 212


,qid,query,item_id
0,10853,I need a product that lasts longer than two we...,B0C6KQP8B3
1,20844,I need a thermometer that works great and has ...,B0BNM9KCF6
2,3197,I need candles that are perfect for weddings. ...,B09ZYBKLNS


In [5]:
t = time.time()
gold = set(queries["item_id"])
catalogue = build_catalogue(gold)
print(f"catalogue: {len(catalogue):,} items ({len(gold):,} gold + {len(catalogue) - len(gold):,} distractors) "
      f"in {time.time() - t:.0f}s")

missing = gold - set(catalogue["item_id"])
assert not missing, f"{len(missing)} gold items not in the catalogue"
check_no_leakage(queries, catalogue)
print("✓ every gold item is in the catalogue; ✓ no ori_review in the index")
catalogue.head(3)

catalogue: 32,614 items (998 gold + 31,616 distractors) in 0s
✓ every gold item is in the catalogue; ✓ no ori_review in the index


,item_id,category,title,text
0,B07NRD63N7,Clothing,Skirt Sports Women's Kelly Bra,Skirt Sports Women's Kelly Bra. Skirt Sports -...
1,B005EDIEDI,Toys,"Axial AXA1202 O-Ring, 3.5x2mm","Axial AXA1202 O-Ring, 3.5x2mm. These are the A..."
2,B08G4XMSV4,Electronics,Beeyoka Privacy Screen Protector Compatible wi...,Beeyoka Privacy Screen Protector Compatible wi...


In [6]:
dev, test = split_queries(queries)
print(f"dev: {len(dev)} queries · test: {len(test)} queries (QUICK_RUN={config.QUICK_RUN})")

lengths = catalogue["text"].str.split().str.len()
print(f"item text: mean {lengths.mean():.0f} words, median {lengths.median():.0f}, max {lengths.max():,}")
catalogue["category"].value_counts().rename("items").to_frame().T

dev: 200 queries · test: 800 queries (QUICK_RUN=False)


item text: mean 86 words, median 32, max 12,237


category,Home,Clothing,Electronics,Tools,Household,Care,Kindle,Pet,Automotive,Garden,...,Instruments,Appliances,Handmade,CDs,Fashion,Health,Beauty,Software,Gift,Magazine
items,5251,4445,2420,2219,1994,1928,1705,1404,1345,1274,...,205,151,61,59,56,24,23,20,8,4


**Data quality.** Some catalogue items have little or no text (e.g. `"Her Leaves Remain. ."`). Queries whose gold item is empty are **unanswerable** by any retriever. We keep them in the test set (dropping them would flatter the numbers) and report them here and in the error analysis.

In [7]:
words = catalogue.set_index("item_id")["text"].str.split().str.len()
gold_words = words.loc[list(gold)]
print(f"gold items: {len(gold)} (2 queries share a gold item)")
print(f"gold items with empty text: {(gold_words == 0).sum()} · with ≤ 5 words: {(gold_words <= 5).sum()}")
print(f"test queries with an empty gold item: {sum(words[i] == 0 for i in test['item_id'])} of {len(test)}")

gold items: 998 (2 queries share a gold item)
gold items with empty text: 4 · with ≤ 5 words: 15
test queries with an empty gold item: 4 of 800


## 4. Baseline

_Phase 2 · BM25 and dense retrieval on the raw request (Module 3 · RAG_Pipeline_Lab)._

## 5. System

### 5a. Retrieval / RAG

_Phase 2 · hybrid (BM25 + dense, RRF) + cross-encoder re-ranker; Recall@10, MRR, nDCG@10 (Module 3 · RAG_Pipeline_Lab)._

### 5b. Agents and tools

_Phases 3–5 · need analyst (Module 2 · Structured_Generation_Lab), MCP server + local `@tool`s, LangGraph supervisor, memory, `interrupt()` before `add_to_cart` (Module 4 · Multi_Agent_System_End_to_End, MCP_Server_and_Tools_Lab)._

### 5c. Fine-tuning (optional)

_Phase 8 stretch · LoRA need analyst._

## 6. Evaluation

_Phase 7 · one results table: baselines, full system, ablations — same test subset._

## 7. Error Analysis

_Phase 7 · at least 10 categorised failures._

## 8. Demo

_Phase 6 · multi-turn chat with live trace; pauses at the cart for approve / reject._

## 9. Limitations & Ethics

- Where does the system fail?
- Who could be harmed if it is wrong?
- Privacy, bias and misuse considerations.

## 10. References

- Dataset: McAuley-Lab/Amazon-C4 (Hou et al., 2024, *Bridging Language and Items for Retrieval and Recommendation*)
- Models: qwen3:4b (Ollama), BAAI/bge-small-en-v1.5, cross-encoder/ms-marco-MiniLM-L-6-v2
- Course labs reused: see each section
- Tools and libraries: LangGraph, LangChain, MCP, FAISS, rank-bm25, sentence-transformers, Hugging Face datasets